### Bay Area Counties and California Air Basins
This notebook creates three maps:
1. Bay Area counties
2. California air basins
3. Intersection result (county-air basin overlaps)

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
import pandas as pd
from pathlib import Path
import ipywidgets

In [2]:
AREA_CRS = "EPSG:3310"  # NAD83 / California Albers
SQM_PER_SQMI = 2589988.110336

In [3]:
COUNTIES_SHP = Path(r"M:\Data\GIS layers\BayArea_counties\region_county_clp\region_county_clp.shp")
AIR_BASIN_SHP = Path(r"M:\Data\GIS layers\ca_air_basins\CaAirBasin.shp")
OUTPUT_SHP = Path(r"M:\Data\GIS layers\ca_air_basins\bay_area_county_air_basin.shp")

In [4]:
# load shapefiles
counties = gpd.read_file(COUNTIES_SHP)
counties.rename(columns={"fipco": "fipco", "coname": "countyName"}, inplace=True)
air_basins = gpd.read_file(AIR_BASIN_SHP)
air_basins.rename(columns={"NAME": "airBasin"}, inplace=True)

# make crs consistent
if air_basins.crs != counties.crs:
    air_basins = air_basins.to_crs(counties.crs)


#### Explore the geometries

In [5]:
# Map 1 (interactive): California air basins + Bay Area counties

# Build a deterministic color map for county legend + polygons
county_names = sorted(counties["countyName"].dropna().unique().tolist())
county_cmap = plt.get_cmap("Set2", max(len(county_names), 1))
county_color_map = {
    name: mcolors.to_hex(county_cmap(i))
    for i, name in enumerate(county_names)
}

counties_plot = counties.copy()
counties_plot["countyColor"] = counties_plot["countyName"].map(county_color_map)

# Air basins layer (keep built-in legend)
m1 = air_basins.explore(
    column="airBasin",
    cmap="tab20",
    legend=True,
    popup=["airBasin"],
    tooltip=["airBasin"],
    style_kwds={"weight": 0.5, "color": "white", "fillOpacity": 0.7},
    name="California Air Basins",
)

# Counties layer (disable built-in legend to avoid overlap)
counties_plot.explore(
    m=m1,
    color=counties_plot["countyColor"],
    legend=False,
    popup=["countyName"],
    tooltip=["countyName"],
    style_kwds={"weight": 1.0, "color": "black", "fillOpacity": 0.6},
    name="Bay Area Counties",
)

# Custom county legend at top-right (non-overlapping with air basin legend)
county_items_html = "".join(
    [
        (
            "<li style='list-style:none; margin-bottom:4px;'>"
            f"<span style='display:inline-block; width:12px; height:12px; "
            f"background:{county_color_map[name]}; border:1px solid #666; margin-right:8px;'></span>"
            f"{name}</li>"
        )
        for name in county_names
    ]
)

county_legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    left: 20px;
    z-index: 9999;
    background: white;
    border: 2px solid #444;
    border-radius: 6px;
    padding: 10px 12px;
    font-size: 12px;
    max-height: 240px;
    overflow-y: auto;
">
  <div style="font-weight:700; margin-bottom:6px;">Bay Area Counties</div>
  <ul style="padding-left:0; margin:0;">{county_items_html}</ul>
</div>
"""
m1.get_root().html.add_child(folium.Element(county_legend_html))

folium.LayerControl(collapsed=False).add_to(m1)

m1.save(r"M:\Data\GIS layers\ca_air_basins\debug\bay_area_county_air_basin_maps.html")
m1

#### Final methodology

Inspect the maps - the Air District shapes don't have cover the entire counties' shapes, therefore, doing intersection would miss some areas, especially along SR-37. - please see the Alternative Methodologies section in the notebook.
The most comprehensive approach is to perform a union of the air basins and counties shapes, then modify air basins manually.

In [6]:
# only keep air basins that are in or surround the Bay Area counties
print(air_basins['airBasin'].unique())
air_basins_gdf = air_basins.loc[
    air_basins["airBasin"].isin(['North Coast','Sacramento Valley','Lake County','North Central Coast',
                                 'San Joaquin Valley','San Francisco Bay'])][["airBasin", "geometry"]].copy()

# union overlay of counties and air basins
counties_gdf = counties[["fipco", "countyName", "geometry"]].copy()
counties_airBasin_union = gpd.overlay(counties_gdf, air_basins_gdf, how="union")

# Compute area in a projected CRS, then convert m^2 to square miles.
area_crs = "EPSG:3310"  # NAD83 / California Albers
counties_airBasin_union_proj = counties_airBasin_union.to_crs(area_crs)
SQM_PER_SQMI = 2589988.110336
counties_airBasin_union["area"] = counties_airBasin_union_proj.geometry.area / SQM_PER_SQMI

# check test results
counties_airBasin_union.sort_values(["fipco", "area"], ascending=[True, False], inplace=True)
counties_airBasin_union.reset_index(drop=True, inplace=True)
display(counties_airBasin_union)

<ArrowStringArray>
[        'North Coast',   'Northeast Plateau',   'Sacramento Valley',
   'Mountain Counties',         'Lake County',          'Lake Tahoe',
 'Great Basin Valleys',  'San Joaquin Valley', 'North Central Coast',
       'Mojave Desert', 'South Central Coast',          'Salton Sea',
   'San Francisco Bay',         'South Coast',    'San Diego County']
Length: 15, dtype: str


c:\Users\ywang\AppData\Local\anaconda3\envs\rawfood\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 3672 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


,fipco,countyName,airBasin,geometry,area
0,001,Alameda County,San Francisco Bay,"MULTIPOLYGON (((-122.08156 37.49993, -122.0855...",743.854500
1,001,Alameda County,NaN,"MULTIPOLYGON (((-122.08454 37.49947, -122.0854...",0.574496
2,001,Alameda County,San Joaquin Valley,"MULTIPOLYGON (((-121.55047 37.53934, -121.5501...",0.177702
3,013,Contra Costa County,San Francisco Bay,"MULTIPOLYGON (((-121.57567 38.00297, -121.5748...",729.785112
4,013,Contra Costa County,NaN,"MULTIPOLYGON (((-122.35918 37.89562, -122.3592...",2.861481
5,013,Contra Costa County,San Joaquin Valley,"MULTIPOLYGON (((-121.56682 38.01871, -121.5666...",0.101183
6,041,Marin County,San Francisco Bay,"MULTIPOLYGON (((-122.4267 37.87037, -122.42668...",524.750712
7,041,Marin County,NaN,"MULTIPOLYGON (((-122.42657 37.87036, -122.4264...",0.687572
8,041,Marin County,North Coast,"MULTIPOLYGON (((-123.00189 38.29629, -123.0017...",0.025804
9,055,Napa County,San Francisco Bay,"POLYGON ((-122.1033 38.51329, -122.10331 38.51...",787.637281


In [8]:
# Map 2 (interactive): click polygons for county-air basin popup
union_plot = counties_airBasin_union.copy()
union_plot["airBasin"] = union_plot["airBasin"].fillna("N/A")
union_plot["legendLabel"] = union_plot["countyName"] + " | " + union_plot["airBasin"]

# Keep interactive labels concise and readable.
union_plot["area_sqmi"] = union_plot["area"].round(3)

m2 = union_plot.explore(
    column="legendLabel",
    cmap="Set3",
    legend=False,
    popup=["countyName", "airBasin", "area_sqmi"],
    tooltip=["countyName", "airBasin"],
    style_kwds={"weight": 0.4, "color": "black", "fillOpacity": 0.25},
    name="County-Air Basin Overlaps"
 )

# Add county outlines as an overlay layer.
counties_gdf.boundary.explore(
    m=m2,
    color="black",
    style_kwds={"weight": 1.0},
    name="County Boundaries"
 )

m2.save(r"M:\Data\GIS layers\ca_air_basins\debug\bay_area_county_air_basin_union.html")
m2

Examine county / air basin boundaries more closely
In theory, only Sonoma County and Solano County fall into two air basins each, but the overlay results show 
most counties intersecting with multiple air basins, due to slight boundary misalignments between counties geometry and air basin geometrics, for example, east side of Santa Clara County intersecting with San Joaquin Valley air basin, which is not expected.
Therefore, we do some manual adjustments for Solano and Sonoma counties, and use raw county shapes for other counties:
  - for Sonoma, re-assign non-San Francisco Bay air basin geometries to North Coast air basin, re-assign N/A to San Francisco air basin. Most of the N/A area is in the bay, therefore assigning to SF air basin is appropriate. However, a small portion of N/A is actually along the Pacific Ocean near Gualala Point and should be assign to North Coast, but it won't affect link tagging therefore ignored here for simplicity.
  - for Solano, reassign non-San Francisco Bay air basin geometries to Sacramento Valley air basin, re-assign N/A to San Francisco air basin.  Most of the N/A area is in the bay, therefore assigning to SF air basin is appropriate. However, a very small sliver is along the Sacramento River. For simplicity, assign all N/A to SF air basin because the impact on link tagging is negligible.

In [9]:
# select the two counties
union_adj = counties_airBasin_union.loc[counties_airBasin_union['countyName'].isin(['Solano County', 'Sonoma County'])]

# Solano County
union_adj.loc[
    (union_adj['countyName'] == 'Solano County') & (union_adj['airBasin'].notnull()) & (union_adj['airBasin'] != 'San Francisco Bay'), 'airBasin'] = 'Sacramento Valley'
union_adj.loc[
    (union_adj['countyName'] == 'Solano County') & (union_adj['airBasin'].isnull()), 'airBasin'] = 'San Francisco Bay'

# Sonoma County
union_adj.loc[
    (union_adj['countyName'] == 'Sonoma County') & (union_adj['airBasin'].notnull()) & (union_adj['airBasin'] != 'San Francisco Bay'), 'airBasin'] = 'North Coast'
union_adj.loc[
    (union_adj['countyName'] == 'Sonoma County') & (union_adj['airBasin'].isnull()), 'airBasin'] = 'San Francisco Bay'

union_adj.sort_values(["fipco", "area"], ascending=[True, False], inplace=True)
union_adj.reset_index(drop=True, inplace=True)
display(union_adj)

# dissolve by county and air basin
union_adj_dissolved = union_adj.drop(columns=['fipco']).dissolve(
    by=["countyName", "airBasin"], aggfunc='sum', as_index=False)
display(union_adj_dissolved)

,fipco,countyName,airBasin,geometry,area
0,095,Solano County,Sacramento Valley,"POLYGON ((-121.87984 38.53799, -121.87765 38.5...",470.304626
1,095,Solano County,San Francisco Bay,"MULTIPOLYGON (((-121.9187 38.05094, -121.91873...",352.667940
2,095,Solano County,San Francisco Bay,"MULTIPOLYGON (((-121.91877 38.05082, -121.9188...",19.896090
3,097,Sonoma County,North Coast,"POLYGON ((-123.01506 38.85202, -123.01174 38.8...",971.453224
4,097,Sonoma County,San Francisco Bay,"MULTIPOLYGON (((-122.6238 38.66439, -122.62371...",616.712699
5,097,Sonoma County,North Coast,"MULTIPOLYGON (((-122.82161 38.85015, -122.8210...",0.543549
6,097,Sonoma County,San Francisco Bay,"MULTIPOLYGON (((-123.42979 38.67845, -123.4298...",0.164873


,countyName,airBasin,geometry,area
0,Solano County,Sacramento Valley,"POLYGON ((-121.87984 38.53799, -121.87765 38.5...",470.304626
1,Solano County,San Francisco Bay,"MULTIPOLYGON (((-122.19403 38.05983, -122.1942...",372.564030
2,Sonoma County,North Coast,"POLYGON ((-123.00525 38.85128, -123.00112 38.8...",971.996774
3,Sonoma County,San Francisco Bay,"MULTIPOLYGON (((-123.42919 38.67702, -123.4292...",616.877573


In [10]:
# QAQC - county sizes didn't change
size_unioned = union_adj.groupby("countyName")["area"].sum().reset_index()
counties_proj = counties.to_crs(area_crs)
SQM_PER_SQMI = 2589988.110336
counties["area"] = counties_proj.geometry.area / SQM_PER_SQMI
size_comparison = size_unioned.merge(counties[["countyName", "area"]], on="countyName", suffixes=("_union", "_counties"))
size_comparison["area_diff"] = size_comparison["area_union"] - size_comparison["area_counties"]
size_comparison

,countyName,area_union,area_counties,area_diff
0,Solano County,842.868656,842.868675,-1.952532e-05
1,Sonoma County,1588.874346,1588.874346,4.067558e-07


In [11]:
# join back to other counties
all_counties  = pd.concat([
    union_adj_dissolved,
    counties.loc[~counties["countyName"].isin(['Solano County','Sonoma County'])][['countyName', 'geometry', 'area']]], ignore_index=True)
all_counties_gdf = gpd.GeoDataFrame(all_counties, geometry="geometry", crs=counties.crs)
all_counties_gdf["airBasin"] = all_counties_gdf["airBasin"].fillna("San Francisco Bay")
display(all_counties_gdf.sort_values(["countyName", "airBasin"], ascending=[True, True]))

,countyName,airBasin,geometry,area
4,Alameda County,San Francisco Bay,"MULTIPOLYGON (((-122.0808 37.50053, -122.0826 ...",744.606699
5,Contra Costa County,San Francisco Bay,"MULTIPOLYGON (((-122.35912 37.89563, -122.3591...",732.747777
6,Marin County,San Francisco Bay,"MULTIPOLYGON (((-122.42679 37.87039, -122.4266...",525.464088
7,Napa County,San Francisco Bay,"POLYGON ((-122.10329 38.51335, -122.1033 38.51...",788.576811
8,San Francisco County,San Francisco Bay,"MULTIPOLYGON (((-123.00361 37.69325, -123.0041...",47.529127
9,San Mateo County,San Francisco Bay,"MULTIPOLYGON (((-122.33723 37.10924, -122.3371...",453.440114
10,Santa Clara County,San Francisco Bay,"MULTIPOLYGON (((-122.09688 37.45856, -122.0967...",1298.731738
0,Solano County,Sacramento Valley,"POLYGON ((-121.87984 38.53799, -121.87765 38.5...",470.304626
1,Solano County,San Francisco Bay,"MULTIPOLYGON (((-122.19403 38.05983, -122.1942...",372.564030
2,Sonoma County,North Coast,"POLYGON ((-123.00525 38.85128, -123.00112 38.8...",971.996774


In [ ]:
# create a new column "ctAirBasin" by concatenating county name and air basin name, which will be used as the unique identifier for each county-air basin combination in the final shapefile
airBasin_map = {
    'San Francisco Bay': 'SF',
    'Sacramento Valley': 'SV',
    'North Coast': 'NC'
}
all_counties_gdf["airBasin"] = all_counties_gdf["airBasin"].map(airBasin_map)
all_counties_gdf["ct"] = all_counties_gdf["countyName"].apply(lambda x: str(x).replace(" County", ""))
all_counties_gdf['ctAirBasin'] = all_counties_gdf.apply(lambda row: f"{row['ct']}_{row['airBasin']}", axis=1)
display(all_counties_gdf.sort_values(["countyName", "airBasin"], ascending=[True, True]))

# write to shapefile with relevant columns
all_counties_gdf[[
    'countyName', 'airBasin', 'ctAirBasin', 'geometry'
]].to_file(OUTPUT_SHP)

,countyName,airBasin,geometry,area,ct,ctAirBasin
4,Alameda County,SF,"MULTIPOLYGON (((-122.0808 37.50053, -122.0826 ...",744.606699,Alameda,Alameda_SF
5,Contra Costa County,SF,"MULTIPOLYGON (((-122.35912 37.89563, -122.3591...",732.747777,Contra Costa,Contra Costa_SF
6,Marin County,SF,"MULTIPOLYGON (((-122.42679 37.87039, -122.4266...",525.464088,Marin,Marin_SF
7,Napa County,SF,"POLYGON ((-122.10329 38.51335, -122.1033 38.51...",788.576811,Napa,Napa_SF
8,San Francisco County,SF,"MULTIPOLYGON (((-123.00361 37.69325, -123.0041...",47.529127,San Francisco,San Francisco_SF
9,San Mateo County,SF,"MULTIPOLYGON (((-122.33723 37.10924, -122.3371...",453.440114,San Mateo,San Mateo_SF
10,Santa Clara County,SF,"MULTIPOLYGON (((-122.09688 37.45856, -122.0967...",1298.731738,Santa Clara,Santa Clara_SF
1,Solano County,SF,"MULTIPOLYGON (((-122.19403 38.05983, -122.1942...",372.564030,Solano,Solano_SF
0,Solano County,SV,"POLYGON ((-121.87984 38.53799, -121.87765 38.5...",470.304626,Solano,Solano_SV
2,Sonoma County,NC,"POLYGON ((-123.00525 38.85128, -123.00112 38.8...",971.996774,Sonoma,Sonoma_NC


In [ ]:
# Map 3 (interactive): click polygons for county-air basin popup
counties_air_basin_gdf_plot = all_counties_gdf.copy()
counties_air_basin_gdf_plot["legendLabel"] = counties_air_basin_gdf_plot["countyName"] + " | " + counties_air_basin_gdf_plot["airBasin"]

# Keep interactive labels concise and readable.
counties_air_basin_gdf_plot["area_sqmi"] = counties_air_basin_gdf_plot["area"].round(3)

m3 = counties_air_basin_gdf_plot.explore(
    column="legendLabel",
    cmap="tab20",
    legend=False,
    popup=["countyName", "airBasin", "area_sqmi"],
    tooltip=["countyName", "airBasin"],
    style_kwds={"weight": 0.4, "color": "black", "fillOpacity": 0.85},
    name="County-Air Basin Overlaps"
 )

# Add county outlines as an overlay layer.
counties_gdf.boundary.explore(
    m=m3,
    color="black",
    style_kwds={"weight": 1.0},
    name="County Boundaries"
 )

m3.save(r"M:\Data\GIS layers\ca_air_basins\debug\bay_area_county_air_basin_union_naive.html")
m3

#### Alternative Methodologies

In [14]:
# Method 1: direct intersections

# make a copy and run overlay to get intersection of counties and air basins

intersection = gpd.overlay(counties_gdf, air_basins_gdf, how="intersection")

# Compute area in a projected CRS, then convert m^2 to square miles.
intersection_proj = intersection.to_crs(AREA_CRS)
intersection["area"] = intersection_proj.geometry.area / SQM_PER_SQMI

# check intersection results
display(intersection.sort_values(["fipco", "area"], ascending=[True, False]))

,fipco,countyName,airBasin,geometry,area
1,001,Alameda County,San Francisco Bay,"MULTIPOLYGON (((-122.08156 37.49993, -122.0855...",743.854500
0,001,Alameda County,San Joaquin Valley,"MULTIPOLYGON (((-121.55047 37.53934, -121.5501...",0.177702
3,013,Contra Costa County,San Francisco Bay,"MULTIPOLYGON (((-121.57567 38.00297, -121.5748...",729.785112
2,013,Contra Costa County,San Joaquin Valley,"MULTIPOLYGON (((-121.56682 38.01871, -121.5666...",0.101183
5,041,Marin County,San Francisco Bay,"MULTIPOLYGON (((-122.4267 37.87037, -122.42668...",524.750712
4,041,Marin County,North Coast,"MULTIPOLYGON (((-123.00189 38.29629, -123.0017...",0.025804
9,055,Napa County,San Francisco Bay,"POLYGON ((-122.1033 38.51329, -122.10331 38.51...",787.637281
8,055,Napa County,Lake County,"MULTIPOLYGON (((-122.62741 38.66751, -122.6217...",0.423262
7,055,Napa County,Sacramento Valley,"MULTIPOLYGON (((-122.10638 38.50701, -122.1064...",0.355146
6,055,Napa County,North Coast,"MULTIPOLYGON (((-122.64472 38.60301, -122.6447...",0.058479


In [15]:
# Map 4 (interactive): click polygons for county-air basin popup
intersection_plot = intersection.copy()
intersection_plot["legendLabel"] = intersection_plot["countyName"] + " | " + intersection_plot["airBasin"]

# Keep interactive labels concise and readable.
intersection_plot["area_sqmi"] = intersection_plot["area"].round(3)

m4 = intersection_plot.explore(
    column="legendLabel",
    cmap="Set3",
    legend=False,
    popup=["countyName", "airBasin", "area_sqmi"],
    tooltip=["countyName", "airBasin"],
    style_kwds={"weight": 0.4, "color": "black", "fillOpacity": 0.85},
    name="County-Air Basin Overlaps"
 )

# Add county outlines as an overlay layer.
counties_gdf.boundary.explore(
    m=m4,
    color="black",
    style_kwds={"weight": 1.0},
    name="County Boundaries"
 )

m4.save(r"M:\Data\GIS layers\ca_air_basins\debug\bay_area_county_air_basin_intersection_map_alt1.html")
m4

In [16]:
# Alternative method 3: only use SF air basin shape
SF_air_basins = air_basins_gdf.loc[air_basins_gdf["airBasin"].isin(["San Francisco Bay"])]
solano_sonoma = counties_gdf.loc[counties_gdf["countyName"].isin(["Solano County", "Sonoma County"])]

# parts of the two counties that fall into SF air basin
intersection_SF = gpd.overlay(
    solano_sonoma,
    SF_air_basins, how="intersection")

# Compute area in a projected CRS, then convert m^2 to square miles.
area_crs = "EPSG:3310"  # NAD83 / California Albers
intersection_SF_proj = intersection_SF.to_crs(area_crs)
SQM_PER_SQMI = 2589988.110336
intersection_SF["area"] = intersection_SF_proj.geometry.area / SQM_PER_SQMI

display(intersection_SF.sort_values(["fipco", "area"], ascending=[True, False]))

# parts of the two counties that do NOT fall into SF air basin
intersection_nonSF = gpd.overlay(solano_sonoma, SF_air_basins, how="difference")
intersection_nonSF = intersection_nonSF.rename(columns={"fipco": "fipco", "coname": "countyName", "NAME": "airBasin"})

# add Air Basin column for non-SF parts
intersection_nonSF.loc[intersection_nonSF['countyName'] == 'Solano County', "airBasin"] = "Sacramento Valley"
intersection_nonSF.loc[intersection_nonSF['countyName'] == 'Sonoma County', "airBasin"] = "North Coast"

# Compute area in a projected CRS, then convert m^2 to square miles.
area_crs = "EPSG:3310"  # NAD83 / California Albers
intersection_nonSF_proj = intersection_nonSF.to_crs(AREA_CRS)
intersection_nonSF["area"] = intersection_nonSF_proj.geometry.area / SQM_PER_SQMI

display(intersection_nonSF.sort_values(["fipco", "area"], ascending=[True, False]))

# verify the total area of county parts equals the total area of the original county
counties_proj = counties.to_crs(AREA_CRS)
counties["area"] = counties_proj.geometry.area / SQM_PER_SQMI

value1 = counties.loc[counties["countyName"] == "Solano County"]["area"].values[0]
value2 = intersection_SF.loc[intersection_SF["countyName"] == "Solano County"]["area"].values[0] + \
    intersection_nonSF.loc[intersection_nonSF["countyName"] == "Solano County"]["area"].values[0]
print(value1)
print(value2)
print(value1 - value2)

value1 = counties.loc[counties["countyName"] == "Sonoma County"]["area"].values[0]
value2 = intersection_SF.loc[intersection_SF["countyName"] == "Sonoma County"]["area"].values[0] + \
    intersection_nonSF.loc[intersection_nonSF["countyName"] == "Sonoma County"]["area"].values[0]
print(value1 - value2)

counties_air_basin = pd.concat([
    intersection_SF,
    intersection_nonSF,
    counties.loc[~counties["countyName"].isin(['Solano County','Sonoma County'])]], ignore_index=True)
counties_air_basin_gdf = gpd.GeoDataFrame(counties_air_basin, geometry="geometry", crs=counties.crs)
counties_air_basin_gdf["airBasin"] =counties_air_basin_gdf["airBasin"].fillna("San Francisco Bay")
display(counties_air_basin_gdf.sort_values(["countyName", "airBasin"], ascending=[True, True]))

,fipco,countyName,airBasin,geometry,area
0,095,Solano County,San Francisco Bay,"MULTIPOLYGON (((-121.9187 38.05094, -121.91873...",352.667940
1,097,Sonoma County,San Francisco Bay,"MULTIPOLYGON (((-122.6238 38.66439, -122.62371...",616.712699


,fipco,countyName,geometry,airBasin,area
0,095,Solano County,"MULTIPOLYGON (((-121.91877 38.05082, -121.9188...",Sacramento Valley,490.200715
1,097,Sonoma County,"MULTIPOLYGON (((-123.40852 38.6542, -123.40857...",North Coast,972.161647


842.8686750562417
842.8686554994351
1.9556806591936038e-05
-3.4790741665347014e-07


,fipco,countyName,airBasin,geometry,area,fipst
4,001,Alameda County,San Francisco Bay,"MULTIPOLYGON (((-122.0808 37.50053, -122.0826 ...",744.606699,06
5,013,Contra Costa County,San Francisco Bay,"MULTIPOLYGON (((-122.35912 37.89563, -122.3591...",732.747777,06
6,041,Marin County,San Francisco Bay,"MULTIPOLYGON (((-122.42679 37.87039, -122.4266...",525.464088,06
7,055,Napa County,San Francisco Bay,"POLYGON ((-122.10329 38.51335, -122.1033 38.51...",788.576811,06
8,075,San Francisco County,San Francisco Bay,"MULTIPOLYGON (((-123.00361 37.69325, -123.0041...",47.529127,06
9,081,San Mateo County,San Francisco Bay,"MULTIPOLYGON (((-122.33723 37.10924, -122.3371...",453.440114,06
10,085,Santa Clara County,San Francisco Bay,"MULTIPOLYGON (((-122.09688 37.45856, -122.0967...",1298.731738,06
2,095,Solano County,Sacramento Valley,"MULTIPOLYGON (((-121.91877 38.05082, -121.9188...",490.200715,NaN
0,095,Solano County,San Francisco Bay,"MULTIPOLYGON (((-121.9187 38.05094, -121.91873...",352.667940,NaN
3,097,Sonoma County,North Coast,"MULTIPOLYGON (((-123.40852 38.6542, -123.40857...",972.161647,NaN


In [17]:
# Map 5 (interactive): click polygons for county-air basin popup
counties_air_basin_gdf_plot = counties_air_basin_gdf.copy()
counties_air_basin_gdf_plot["legendLabel"] = counties_air_basin_gdf_plot["countyName"] + " | " + counties_air_basin_gdf_plot["airBasin"]

# Keep interactive labels concise and readable.
counties_air_basin_gdf_plot["area_sqmi"] = counties_air_basin_gdf_plot["area"].round(3)

m5 = counties_air_basin_gdf_plot.explore(
    column="legendLabel",
    cmap="tab20",
    legend=False,
    popup=["countyName", "airBasin", "area_sqmi"],
    tooltip=["countyName", "airBasin"],
    style_kwds={"weight": 0.4, "color": "black", "fillOpacity": 0.85},
    name="County-Air Basin Overlaps"
 )

# Add county outlines as an overlay layer.
counties_gdf.boundary.explore(
    m=m5,
    color="black",
    style_kwds={"weight": 1.0},
    name="County Boundaries"
 )

m5.save(r"M:\Data\GIS layers\ca_air_basins\debug\bay_area_county_air_basin_intersection_map_alt2.html")
m5